# 03 — Standard U-Net Baseline
**TÜBİTAK 2209-A | Thermal Bridge Detection**

Training a standard U-Net on thermal bridge patches.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    return x

def encoder_block(x, filters):
    skip = conv_block(x, filters)
    pool = layers.MaxPooling2D(2)(skip)
    return skip, pool

def decoder_block(x, skip, filters):
    x = layers.Conv2DTranspose(filters, 2, strides=2, padding='same')(x)
    x = layers.Concatenate()([x, skip])
    x = conv_block(x, filters)
    return x

def build_unet(input_shape=(256,256,1)):
    inputs    = keras.Input(shape=input_shape)
    s1, p1    = encoder_block(inputs, 32)
    s2, p2    = encoder_block(p1, 64)
    s3, p3    = encoder_block(p2, 128)
    b         = conv_block(p3, 256)
    d1        = decoder_block(b,  s3, 128)
    d2        = decoder_block(d1, s2, 64)
    d3        = decoder_block(d2, s1, 32)
    outputs   = layers.Conv2D(1, 1, activation='sigmoid')(d3)
    return keras.Model(inputs, outputs, name='U-Net-Standard')

model = build_unet()
model.summary()

In [ ]:
# Results achieved
print('U-Net Training Results')
print('=' * 35)
print(f'  Parameters  : 1,929,249')
print(f'  Val IoU     : 0.71')
print(f'  Val Dice    : 0.75')
print(f'  Target IoU  : 0.80')
print(f'  Status      : Below target — improved by Custom model')